In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("PeakMedia_DataCleaning").getOrCreate()
from pyspark.sql.functions import col, sum, count, isnan, when, lit, to_date, substring, concat, regexp_replace
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, 2988ee3a-c578-4784-9c75-b76cd1a67e42, 3, Finished, Available, Finished, False)

In [13]:
# -------------------------------------------------------------
# STEP 1 — Define paths to Raw CSV files in the Lakehouse
# -------------------------------------------------------------
path_trackaff = "Files/Raw_Data/trackAff_feed.csv"
path_affmatrix = "Files/Raw_Data/affMatrix_feed.csv"
path_campaign  = "Files/Raw_Data/campaign_data.csv"

# -------------------------------------------------------------
# STEP 2 — Read the raw CSVs as DataFrames
# -------------------------------------------------------------
trackAff_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path_trackaff)
)

affMatrix_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path_affmatrix)
)

campaign_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path_campaign)
)

# Quick schema check (optional)
trackAff_df.printSchema()
affMatrix_df.printSchema()
campaign_df.printSchema()

# -------------------------------------------------------------
# STEP 3 — Write DataFrames back as Parquet files (raw zone)
# -------------------------------------------------------------
trackAff_df.write.mode("overwrite").parquet("Files/Raw_Parquet/trackAff_raw")
affMatrix_df.write.mode("overwrite").parquet("Files/Raw_Parquet/affMatrix_raw")
campaign_df.write.mode("overwrite").parquet("Files/Raw_Parquet/campaign_raw")

print("✅ All raw CSVs successfully written as Parquet files under 'Files/Raw_Parquet/'")


StatementMeta(, 91f8fced-78b9-4aa1-a12b-ddef292adc23, 15, Finished, Available, Finished, False)

root
 |-- date: string (nullable = true)
 |-- operator_name: string (nullable = true)
 |-- tracking_id: long (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- registrations: double (nullable = true)
 |-- ftds: double (nullable = true)
 |-- commission_per_ftd: integer (nullable = true)
 |-- total_commission: double (nullable = true)
 |-- currency: string (nullable = true)

root
 |-- player_id: string (nullable = true)
 |-- operator_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- tracking_id: long (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- ftd_date: string (nullable = true)
 |-- commission_per_ftd: integer (nullable = true)
 |-- commission_earned: integer (nullable = true)
 |-- currency: string (nullable = true)

root
 |-- date: date (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- impressions: integer (nullable = true

In [14]:
# ================================================================
# STEP 1 — Load Parquet Files (no need for CSVs anymore)
# ================================================================
trackAff_df = spark.read.parquet("Files/Raw_Parquet/trackAff_raw")
affMatrix_df = spark.read.parquet("Files/Raw_Parquet/affMatrix_raw")
campaign_df  = spark.read.parquet("Files/Raw_Parquet/campaign_raw")

# Optional: count rows and print simple schema
print("TrackAff rows:", trackAff_df.count())
print("AffMatrix rows:", affMatrix_df.count())
print("Campaign rows:", campaign_df.count())


StatementMeta(, 91f8fced-78b9-4aa1-a12b-ddef292adc23, 16, Finished, Available, Finished, False)

TrackAff rows: 814
AffMatrix rows: 4066
Campaign rows: 270


In [15]:
# ================================================================
# STEP 2 — Normalize Column Names (consistent lowercase + underscores)
# ================================================================
def normalize_cols(df):
    for colname in df.columns:
        df = df.withColumnRenamed(colname, colname.strip().lower().replace(" ", "_"))
    return df

trackAff_df = normalize_cols(trackAff_df)
affMatrix_df = normalize_cols(affMatrix_df)
campaign_df  = normalize_cols(campaign_df)


StatementMeta(, 91f8fced-78b9-4aa1-a12b-ddef292adc23, 17, Finished, Available, Finished, False)

In [2]:
# ==============================================================
# Cleanse campaign_data
# ==============================================================

# Load campaign parquet created earlier
campaign_df = spark.read.parquet("Files/Raw_Parquet/campaign_raw")

# --- Step 1: Ensure date parsed and reformatted to dd/MM/yyyy ---
campaign_df = campaign_df.withColumn(
    "date",
    F.date_format(F.to_date(F.col("date"), "yyyy-MM-dd"), "dd/MM/yyyy")
)

# --- Step 2: Pad campaign_id with leading zeros to 3 digits ---
campaign_df = campaign_df.withColumn(
    "campaign_id",
    F.lpad(F.col("campaign_id").cast("string"), 3, "0")
)

# Optional: sanity check
campaign_df.select("date", "campaign_id", "campaign_name").show(5, False)

# ==============================================================
#Cleanse trackAffdf
# ==============================================================
trackAff_df = spark.read.parquet("Files/Raw_Parquet/trackAff_raw")
# 1. Drop records where tracking_id is NULL
trackAff_df = trackAff_df.filter(F.col("tracking_id").isNotNull())

# 2. Registrations: if NULL or < 0 → set to 0
trackAff_df = trackAff_df.withColumn(
    "registrations",
    F.when((F.col("registrations").isNull()) | (F.col("registrations") < 0), F.lit(0))
     .otherwise(F.col("registrations"))
)

# 3. Derive date from tracking_id as yyyy-MM-dd
trackAff_df = trackAff_df.withColumn(
    "date",
    F.concat(
        F.lit("20"),
        F.substring(F.col("tracking_id").cast("string"), 1, 2),
        F.lit("-"),
        F.substring(F.col("tracking_id").cast("string"), 3, 2),
        F.lit("-"),
        F.substring(F.col("tracking_id").cast("string"), 5, 2)
    )
)

# 4. FTDs: if NULL → set to 0
trackAff_df = trackAff_df.withColumn(
    "ftds",
    F.when(F.col("ftds").isNull(), F.lit(0)).otherwise(F.col("ftds"))
)

# 5. Total Commission: if NULL → set to 0
trackAff_df = trackAff_df.withColumn(
    "total_commission",
    F.when(F.col("total_commission").isNull(), F.lit(0))
     .otherwise(F.col("total_commission"))
)

# ==============================================================
# Cleanse affMatrix_feed (retain original column names)
# ==============================================================

affMatrix_df = spark.read.parquet("Files/Raw_Parquet/affMatrix_raw")

# 1. Remove records where player_id or tracking_id is NULL
affMatrix_df = affMatrix_df.filter(
    (F.col("player_id").isNotNull()) & (F.col("tracking_id").isNotNull())
)

# 2. Fill NULL product values as "Default"
affMatrix_df = affMatrix_df.fillna({"product": "Default"})

# 3. Convert ftddate and registrationdate to proper date format (using dd-MM-yyyy)
affMatrix_df = affMatrix_df.withColumn(
    "ftd_date", F.to_date(F.col("ftd_date"), "dd-MM-yyyy")
).withColumn(
    "registration_date", F.to_date(F.col("registration_date"), "dd-MM-yyyy")
)

# 4. Check if ftd_date is greater than or equal to registration_date
affMatrix_df = affMatrix_df.filter(
    (F.col("ftd_date").isNull()) | (F.col("ftd_date") >= F.col("registration_date"))
)

# 5. Replace NULL ftd_date with default 01-01-1900
affMatrix_df = affMatrix_df.withColumn(
    "ftd_date",
    F.when(F.col("ftd_date").isNull(), F.lit("1900-01-01").cast("date"))
     .otherwise(F.col("ftd_date"))
)

# 6. If commission_earned is negative, set to 0
affMatrix_df = affMatrix_df.withColumn(
    "commission_earned",
    F.when(F.col("commission_earned") < 0, F.lit(0)).otherwise(F.col("commission_earned"))
)

StatementMeta(, 2988ee3a-c578-4784-9c75-b76cd1a67e42, 4, Finished, Available, Finished, False)

+----------+-----------+----------------------+
|date      |campaign_id|campaign_name         |
+----------+-----------+----------------------+
|01/01/2025|001        |Spring_Slots_Push     |
|01/01/2025|002        |Welcome_Bonus_Drive   |
|01/01/2025|003        |Champions_Sports_Promo|
|02/01/2025|001        |Spring_Slots_Push     |
|02/01/2025|002        |Welcome_Bonus_Drive   |
+----------+-----------+----------------------+
only showing top 5 rows


In [17]:
# ================================================================
# STEP 6 — Write Out Cleaned DataFrames as Lakehouse Tables
# ================================================================

trackAff_df.write.format("delta").mode("overwrite").saveAsTable("dbo.trackAff_clean")
affMatrix_df.write.format("delta").mode("overwrite").saveAsTable("dbo.affMatrix_clean")
campaign_df.write.format("delta").mode("overwrite").saveAsTable("dbo.campaign_clean")

print("✅ Cleaned data written successfully as Lakehouse tables.")

StatementMeta(, 91f8fced-78b9-4aa1-a12b-ddef292adc23, 19, Finished, Available, Finished, False)

✅ Cleaned data written successfully as Lakehouse tables.


In [4]:
# ==========================================================
# DIM_OPERATOR
# ==========================================================
operator_union = affMatrix_df.select("operator_name").unionByName(
    trackAff_df.select("operator_name")
).distinct()

dim_operator_df = operator_union.withColumn(
    "operator_id",
    F.row_number().over(Window.orderBy("operator_name"))
).select("operator_id", "operator_name")

# Write to the existing Warehouse table
dim_operator_df.write.mode("overwrite").saveAsTable("dbo.dim_operator")

# ==========================================================
# DIM_CAMPAIGN
# ==========================================================
dim_campaign_df = (
    campaign_df
    .select("campaign_id", "campaign_name")
    .distinct()
)

dim_campaign_df.write.mode("overwrite").saveAsTable("dbo.dim_campaign")

# ==========================================================
# DIM_CARD
# ==========================================================
card_union = affMatrix_df.select("operator_name").unionByName(
    trackAff_df.select("operator_name")
).distinct()

dim_card_df = card_union.withColumn(
    "card_id",
    F.row_number().over(Window.orderBy("operator_name"))
).select("card_id", "operator_name")

dim_card_df.write.mode("overwrite").saveAsTable("dbo.dim_card")

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================================
# DIM_CURRENCY
# ==========================================================
currency_union = (
    affMatrix_df
        .select(F.trim(F.col("currency")).alias("currency"))
    .unionByName(
        trackAff_df.select(F.trim(F.col("currency")).alias("currency"))
    )
    .filter(F.col("currency").isNotNull())
    .filter(F.col("currency") != "")
    .distinct()
)

dim_currency_df = (
    currency_union
    .withColumn(
        "currency_id",
        F.row_number().over(Window.orderBy("currency"))
    )
    .select("currency_id", "currency")
)

# Write to lakehouse table
dim_currency_df.write.mode("overwrite").saveAsTable("LH_PeakMedia.dbo.dim_currency")

StatementMeta(, 2988ee3a-c578-4784-9c75-b76cd1a67e42, 6, Finished, Available, Finished, False)